# Analyse NLP des Rapports de Collecte
Objectif : Prédire la catégorie du déchet en se basant uniquement sur les descriptions textuelles fournies dans `Rapport_Collecte`.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

## Chargement des données textuelles


In [ ]:
train = pd.read_csv('../data/processed/train.csv')
val = pd.read_csv('../data/processed/val.csv')

# Nettoyage basique : suppression des NaN dans le texte
train['Rapport_Collecte'] = train['Rapport_Collecte'].fillna('N/A')
val['Rapport_Collecte'] = val['Rapport_Collecte'].fillna('N/A')

X_train_text = train['Rapport_Collecte']
y_train = train['Categorie']
X_val_text = val['Rapport_Collecte']
y_val = val['Categorie']

## Comparaison Vectoriseurs x Classifieurs
Nous testons systématiquement le Bag of Words (BoW) et le TF-IDF avec différents algorithmes pour trouver la meilleure combinaison.


In [ ]:
vectorizers = {
    'BoW': CountVectorizer(max_features=5000, stop_words=None),
    'TF-IDF': TfidfVectorizer(max_features=5000, stop_words=None)
}

classifiers = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Linear SVC': LinearSVC()
}

nlp_results = []

for vec_name, vec in vectorizers.items():
    X_train_vec = vec.fit_transform(X_train_text)
    X_val_vec = vec.transform(X_val_text)
    
    for clf_name, clf in classifiers.items():
        clf.fit(X_train_vec, y_train)
        preds = clf.predict(X_val_vec)
        
        acc = accuracy_score(y_val, preds)
        f1 = f1_score(y_val, preds, average='weighted')
        
        nlp_results.append({
            'Vectorizer': vec_name,
            'Classifier': clf_name,
            'Accuracy': acc,
            'F1-Score': f1,
            'Model': clf,
            'Vec': vec
        })
        print(f"{vec_name} + {clf_name:<20} -> Acc: {acc:.4f}")

## Synthèse des résultats NLP


In [ ]:
df_nlp = pd.DataFrame(nlp_results)
fig_nlp = px.bar(
    df_nlp, 
    x='Classifier', y='Accuracy', color='Vectorizer', 
    barmode='group', title='Performance des combinaisons NLP',
    height=500
)
fig_nlp.show()

display(df_nlp[['Vectorizer', 'Classifier', 'Accuracy', 'F1-Score']].sort_values('Accuracy', ascending=False).round(4))

## Sauvegarde du meilleur modèle NLP


In [ ]:
best_nlp_idx = df_nlp['Accuracy'].idxmax()
best_nlp_model = df_nlp.loc[best_nlp_idx, 'Model']
best_vec = df_nlp.loc[best_nlp_idx, 'Vec']

print(f"Meilleure combinaison : {df_nlp.loc[best_nlp_idx, 'Vectorizer']} + {df_nlp.loc[best_nlp_idx, 'Classifier']}")

os.makedirs('../models', exist_ok=True)
joblib.dump(best_nlp_model, '../models/best_nlp_model.joblib')
joblib.dump(best_vec, '../models/nlp_vectorizer.joblib')
print("Modèle et Vectoriseur sauvegardés dans ../models/")

## Conclusion
L'utilisation de techniques NLP sur les rapports de collecte permet d'extraire une information précieuse qui complète les mesures physiques. Une approche hybride (fusionnant Features Physiques + Textuelles) pourrait encore améliorer la précision globale du système.
